<div style="background:linear-gradient(135deg,#0b1120,#1a2540);padding:40px 30px;border-radius:16px;border-left:6px solid #2dd4bf;margin-bottom:10px">
  <h1 style="color:#e0e7f0;font-family:sans-serif;margin:0 0 8px 0">🏨 Hotel Booking Analysis</h1>
  <h3 style="color:#2dd4bf;font-family:sans-serif;margin:0 0 12px 0">Cancellation Prediction & Market Intelligence — EDA Dashboard</h3>
  <p style="color:#64748b;font-family:sans-serif;margin:0">Exploratory Data Analysis · Cancellations · Revenue · ADR · Lead Time · Market Segments · Guest Behaviour</p>
</div>

## ⚙️ Section 1 — Setup & Configuration

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly -q
print('✅ Libraries ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Dashboard Theme ───────────────────────────────────────────
DARK_BG  = '#0b1120'
SURFACE  = '#131c2e'
SURFACE2 = '#1a2540'
BORDER   = '#1e2e4a'
TEXT     = '#e0e7f0'
MUTED    = '#64748b'
TEAL     = '#2dd4bf'
BLUE     = '#3b82f6'
PURPLE   = '#8b5cf6'
PINK     = '#ec4899'
ORANGE   = '#f97316'
GREEN    = '#22c55e'
YELLOW   = '#eab308'
RED      = '#ef4444'
CYAN     = '#06b6d4'

PALETTE = [TEAL,BLUE,PURPLE,PINK,ORANGE,GREEN,YELLOW,RED,CYAN,
           '#a78bfa','#fb923c','#34d399','#f472b6','#60a5fa','#fbbf24']

plt.rcParams.update({
    'figure.facecolor': DARK_BG,  'axes.facecolor': SURFACE,
    'axes.edgecolor':   BORDER,   'axes.labelcolor': TEXT,
    'axes.titlecolor':  TEXT,     'axes.titlesize': 12,
    'axes.titleweight': 'bold',   'axes.titlepad': 10,
    'axes.grid': True,            'grid.color': BORDER,
    'grid.linewidth': 0.6,        'xtick.color': MUTED,
    'ytick.color': MUTED,         'text.color': TEXT,
    'legend.facecolor': SURFACE2, 'legend.edgecolor': BORDER,
    'legend.fontsize': 9,         'font.family': 'DejaVu Sans',
    'font.size': 10,
})

def style_ax(ax, title, xlabel='', ylabel=''):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold', pad=8)
    if xlabel: ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.tick_params(colors=MUTED, labelsize=8.5)
    ax.grid(color=BORDER, linewidth=0.4, alpha=0.7)

print('✅ Theme configured.')

## 📂 Section 2 — Load Dataset

In [ ]:
# ── Option A: Upload CSV manually ─────────────────────────────
# from google.colab import files
# uploaded = files.upload()  # upload 'hotel booking.csv'
# df_raw = pd.read_csv('hotel booking.csv')

# ── Option B: Load from GitHub mirror ─────────────────────────
URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/hotel_bookings.csv'
try:
    df_raw = pd.read_csv(URL)
    print(f'✅ Loaded from URL: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
except Exception as e:
    print(f'URL load failed ({e}). Generating synthetic data...')
    np.random.seed(2024)
    N = 12000
    months = ['January','February','March','April','May','June',
              'July','August','September','October','November','December']
    hotels = np.random.choice(['City Hotel','Resort Hotel'], N, p=[0.60,0.40])
    is_canceled = np.array([
        np.random.binomial(1, 0.37 if h=='City Hotel' else 0.27)
        for h in hotels])
    lead_time  = np.random.exponential(80, N).astype(int).clip(0, 500)
    arr_months = np.random.choice(months, N,
        p=[0.05,0.05,0.07,0.08,0.09,0.10,0.12,0.12,0.10,0.09,0.07,0.06])
    market_seg = np.random.choice(
        ['Online TA','Offline TA/TO','Direct','Corporate','Groups','Complementary','Aviation'],
        N, p=[0.47,0.20,0.15,0.10,0.05,0.02,0.01])
    deposit_type  = np.random.choice(['No Deposit','Non Refund','Refundable'], N, p=[0.87,0.11,0.02])
    customer_type = np.random.choice(['Transient','Contract','Transient-Party','Group'], N, p=[0.75,0.10,0.12,0.03])
    reserved_room = np.random.choice(list('ABCDEFGH'), N, p=[0.55,0.05,0.07,0.18,0.07,0.04,0.03,0.01])
    countries     = np.random.choice(['PRT','GBR','FRA','ESP','DEU','ITA','USA','BRA','NLD','RUS'],
                                     N, p=[0.28,0.11,0.10,0.08,0.07,0.06,0.05,0.04,0.03,0.18])
    adults   = np.random.choice([1,2,3,4], N, p=[0.20,0.65,0.12,0.03])
    children = np.random.choice([0,1,2,3], N, p=[0.83,0.10,0.06,0.01])
    week_nts = np.random.choice(range(8), N, p=[0.05,0.10,0.20,0.25,0.18,0.12,0.07,0.03])
    wknd_nts = np.random.choice(range(5), N, p=[0.15,0.25,0.30,0.20,0.10])
    adr = np.clip(80+adults*15+children*10+(lead_time*0.05)+
                  np.where(hotels=='Resort Hotel',20,0)+np.random.normal(0,25,N), 20, 400)
    special_req = np.random.choice([0,1,2,3,4,5], N, p=[0.35,0.32,0.18,0.10,0.04,0.01])
    booking_chg = np.random.choice(range(8), N, p=[0.60,0.20,0.10,0.05,0.02,0.01,0.01,0.01])
    years       = np.random.choice([2015,2016,2017], N, p=[0.15,0.45,0.40])
    is_repeated = np.random.choice([0,1], N, p=[0.97,0.03])
    parking     = np.random.choice([0,1,2], N, p=[0.93,0.06,0.01])
    df_raw = pd.DataFrame({
        'hotel': hotels, 'is_canceled': is_canceled,
        'lead_time': lead_time,
        'arrival_date_year': years, 'arrival_date_month': arr_months,
        'stays_in_week_nights': week_nts, 'stays_in_weekend_nights': wknd_nts,
        'adults': adults, 'children': children.astype(float), 'babies': np.zeros(N),
        'market_segment': market_seg, 'deposit_type': deposit_type,
        'customer_type': customer_type, 'reserved_room_type': reserved_room,
        'country': countries, 'adr': adr,
        'total_of_special_requests': special_req,
        'booking_changes': booking_chg,
        'previous_cancellations': np.random.choice([0,1,2,3], N, p=[0.87,0.08,0.03,0.02]),
        'is_repeated_guest': is_repeated,
        'required_car_parking_spaces': parking,
    })
    print(f'✅ Synthetic dataset: {len(df_raw):,} rows')

df_raw.head(3)

## 🧹 Section 3 — Data Cleaning & Feature Engineering

In [ ]:
df = df_raw.copy()

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# Numeric coercions
df['adr'] = pd.to_numeric(df['adr'], errors='coerce')
df = df[df['adr'] >= 0].copy()

# Derived features
df['total_stays']          = df['stays_in_week_nights'] + df['stays_in_weekend_nights']
df['revenue_per_booking']  = df['adr'] * df['total_stays']
df['total_guests']         = df['adults'] + df['children'].fillna(0) + df['babies'].fillna(0)

# Month ordering
MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
if df['arrival_date_month'].dtype == object:
    df['arrival_date_month'] = pd.Categorical(
        df['arrival_date_month'], categories=MONTH_ORDER, ordered=True)

# Standardise text
df['hotel']          = df['hotel'].str.strip()
df['market_segment'] = df['market_segment'].str.strip()
df['customer_type']  = df['customer_type'].str.strip()
df['deposit_type']   = df['deposit_type'].str.strip()

print(f'✅ Clean dataset: {len(df):,} rows × {df.shape[1]} columns')
print('\nNull values (key columns):')
print(df[['adr','total_stays','lead_time','is_canceled']].isnull().sum())

## 📊 Section 4 — KPI Summary

In [ ]:
total_bookings  = len(df)
total_canceled  = df['is_canceled'].sum()
cancel_rate     = df['is_canceled'].mean() * 100
avg_adr         = df['adr'].mean()
avg_lead        = df['lead_time'].mean()
avg_stay        = df['total_stays'].mean()
free_pct        = (df['required_car_parking_spaces'] > 0).mean() * 100
top_country     = df['country'].value_counts().idxmax()
top_month       = df['arrival_date_month'].value_counts().idxmax()
city_cancel     = df[df['hotel']=='City Hotel']['is_canceled'].mean()*100
resort_cancel   = df[df['hotel']=='Resort Hotel']['is_canceled'].mean()*100
avg_rev_new     = df[df['is_repeated_guest']==0]['total_stays'].mean()
avg_rev_repeat  = df[df['is_repeated_guest']==1]['total_stays'].mean()
avg_rev_free_t  = df[df['is_canceled']==0]['revenue_per_booking'].mean()

print('=' * 58)
print('      🏨  HOTEL BOOKING DASHBOARD  —  KPI SUMMARY')
print('=' * 58)
print(f'  Total Bookings           : {total_bookings:>10,}')
print(f'  Total Cancellations      : {total_canceled:>10,}')
print(f'  Overall Cancel Rate      : {cancel_rate:>9.1f}%')
print(f'  City Hotel Cancel Rate   : {city_cancel:>9.1f}%')
print(f'  Resort Hotel Cancel Rate : {resort_cancel:>9.1f}%')
print(f'  Average ADR (EUR)        : {avg_adr:>9.2f}')
print(f'  Average Lead Time        : {avg_lead:>9.1f} days')
print(f'  Average Stay Duration    : {avg_stay:>9.2f} nights')
print(f'  Top Booking Country      : {top_country}')
print(f'  Avg Stay — New Guest     : {avg_rev_new:>9.2f} nights')
print(f'  Avg Stay — Repeat Guest  : {avg_rev_repeat:>9.2f} nights')
print(f'  Parking Required         : {free_pct:>9.1f}%')
print('=' * 58)

## 📈 Section 5 — Dashboard Page 1: Hotel & Booking Overview

In [ ]:
fig = plt.figure(figsize=(22, 16))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('🏨  Hotel Booking — Overview Dashboard',
             fontsize=18, fontweight='bold', color=TEXT, y=0.98)
fig.text(0.5, 0.963,
         f'{total_bookings:,} total bookings  •  {cancel_rate:.1f}% cancellation rate  •  Avg ADR: €{avg_adr:.0f}',
         ha='center', fontsize=10, color=MUTED)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.35,
                       top=0.94, bottom=0.04, left=0.06, right=0.97)

# ── [0,0] Hotel Type Donut ────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(SURFACE)
htype = df['hotel'].value_counts()
wedges, texts, autos = ax1.pie(
    htype.values, labels=htype.index,
    colors=[TEAL, BLUE], autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor=DARK_BG, linewidth=2),
    pctdistance=0.75)
for t in texts:  t.set_color(TEXT);    t.set_fontsize(10)
for t in autos:  t.set_color(DARK_BG); t.set_fontsize(9); t.set_fontweight('bold')
ax1.set_title('Bookings by Hotel Type', color=TEXT, fontsize=11, fontweight='bold')

# ── [0,1] Cancellation by Hotel ───────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
cb  = df.groupby('hotel')['is_canceled'].mean().reset_index()
bars2 = ax2.bar(cb['hotel'], cb['is_canceled']*100,
                color=[TEAL, PINK], edgecolor=DARK_BG, linewidth=0.4, width=0.5)
for bar in bars2:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
             f'{bar.get_height():.1f}%', ha='center', color=TEXT, fontsize=11, fontweight='bold')
ax2.set_ylim(0, 55)
style_ax(ax2, 'Cancellation Rate by Hotel Type', 'Hotel', 'Cancel Rate (%)')

# ── [0,2] Customer Type Donut ─────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(SURFACE)
ct  = df['customer_type'].value_counts()
wedges3, texts3, autos3 = ax3.pie(
    ct.values, labels=ct.index, colors=PALETTE[:len(ct)],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.5, edgecolor=DARK_BG, linewidth=2),
    pctdistance=0.76)
for t in texts3:  t.set_color(TEXT);    t.set_fontsize(8.5)
for t in autos3:  t.set_color(DARK_BG); t.set_fontsize(8); t.set_fontweight('bold')
ax3.set_title('Customer Type Distribution', color=TEXT, fontsize=11, fontweight='bold')

# ── [1,0:2] Monthly Arrivals ──────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0:2])
mb  = df['arrival_date_month'].value_counts().sort_index()
bars4 = ax4.bar(range(len(mb)), mb.values, color=PALETTE[:12],
                edgecolor=DARK_BG, linewidth=0.3)
ax4.set_xticks(range(len(mb)))
ax4.set_xticklabels([m[:3] for m in MONTH_ORDER[:len(mb)]], fontsize=9)
for bar in bars4:
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
             str(int(bar.get_height())), ha='center', color=TEXT, fontsize=7.5)
style_ax(ax4, 'Monthly Arrivals Distribution', 'Month', 'Bookings')

# ── [1,2] Weeknight vs Weekend ────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
wk  = df['stays_in_week_nights'].value_counts().sort_index().head(8)
wkd = df['stays_in_weekend_nights'].value_counts().sort_index().head(6)
ax5.bar(np.arange(len(wk)) - 0.2, wk.values,  0.38, color=BLUE,
        label='Weeknights', edgecolor=DARK_BG, linewidth=0.3)
ax5.bar(np.arange(len(wkd)) + 0.2, wkd.values, 0.38, color=TEAL,
        label='Weekend Nights', edgecolor=DARK_BG, linewidth=0.3)
ax5.legend()
style_ax(ax5, 'Stay Duration: Weeknights vs Weekend', 'Nights', 'Bookings')

# ── [2,0] Cancellation by Month ───────────────────────────────
ax6 = fig.add_subplot(gs[2, 0:2])
cm_rate = df.groupby('arrival_date_month', observed=True)['is_canceled'].mean()*100
ax6.fill_between(range(len(cm_rate)), cm_rate.values, alpha=0.2, color=RED)
ax6.plot(range(len(cm_rate)), cm_rate.values, color=RED, marker='o',
         markersize=7, linewidth=2.5, markerfacecolor=RED,
         markeredgecolor=DARK_BG, markeredgewidth=1.5)
for x, y in zip(range(len(cm_rate)), cm_rate.values):
    ax6.text(x, y+0.5, f'{y:.1f}%', ha='center', color=TEXT, fontsize=8)
ax6.set_xticks(range(len(cm_rate)))
ax6.set_xticklabels([m[:3] for m in MONTH_ORDER[:len(cm_rate)]], fontsize=9)
ax6.set_ylim(0, cm_rate.max()*1.25)
style_ax(ax6, 'Cancellation Rate by Month', 'Month', 'Cancel Rate (%)')

# ── [2,2] Deposit Type & Cancellation ────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
dep_cancel = df.groupby('deposit_type')['is_canceled'].mean()*100
colors7    = [GREEN if v < 20 else ORANGE if v < 50 else RED for v in dep_cancel.values]
bars7 = ax7.bar(dep_cancel.index, dep_cancel.values,
                color=colors7, edgecolor=DARK_BG, linewidth=0.3, width=0.5)
for bar in bars7:
    ax7.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{bar.get_height():.1f}%', ha='center', color=TEXT, fontsize=10, fontweight='bold')
ax7.tick_params(axis='x', labelsize=8)
style_ax(ax7, 'Cancel Rate by Deposit Type', 'Deposit Type', 'Cancel Rate (%)')

plt.savefig('hotel_dashboard_p1.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Dashboard Page 1 saved.')

## 💰 Section 6 — Dashboard Page 2: Revenue & ADR Analysis

In [ ]:
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('💰  Revenue & Average Daily Rate (ADR) Dashboard',
             fontsize=18, fontweight='bold', color=TEXT, y=0.98)
fig.text(0.5, 0.963, f'Avg ADR: €{avg_adr:.2f}  •  Avg Stay: {avg_stay:.1f} nights',
         ha='center', fontsize=10, color=MUTED)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.94, bottom=0.05, left=0.06, right=0.97)

# ── [0,0:2] ADR by Month ──────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0:2])
adr_m = df.groupby('arrival_date_month', observed=True)['adr'].mean()
ax1.fill_between(range(len(adr_m)), adr_m.values, alpha=0.2, color=BLUE)
ax1.plot(range(len(adr_m)), adr_m.values, color=BLUE, marker='s',
         markersize=7, linewidth=2.5, markerfacecolor=BLUE,
         markeredgecolor=DARK_BG, markeredgewidth=1.5)
for x, y in zip(range(len(adr_m)), adr_m.values):
    ax1.text(x, y+0.5, f'{y:.0f}', ha='center', color=TEXT, fontsize=8.5)
ax1.set_xticks(range(len(adr_m)))
ax1.set_xticklabels([m[:3] for m in MONTH_ORDER[:len(adr_m)]], fontsize=9)
style_ax(ax1, 'Average Daily Rate (ADR) by Month', 'Month', 'ADR (EUR)')

# ── [0,2] ADR by Year ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
if 'arrival_date_year' in df.columns:
    adr_yr = df.groupby('arrival_date_year')['adr'].mean()
    ax2.plot(adr_yr.index, adr_yr.values, color=CYAN, marker='o', markersize=10,
             linewidth=2.5, markerfacecolor=CYAN, markeredgecolor=DARK_BG, markeredgewidth=2)
    for x, y in zip(adr_yr.index, adr_yr.values):
        ax2.text(x, y+0.5, f'{y:.1f}', ha='center', color=TEXT, fontsize=10, fontweight='bold')
    ax2.set_xticks(adr_yr.index)
style_ax(ax2, 'Avg ADR by Year', 'Year', 'ADR (EUR)')

# ── [1,0:2] Revenue by Month ──────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0:2])
rev_m  = df.groupby('arrival_date_month', observed=True)['revenue_per_booking'].sum()/1e6
bars3  = ax3.bar(range(len(rev_m)), rev_m.values,
                 color=PALETTE[:12], edgecolor=DARK_BG, linewidth=0.3)
ax3.set_xticks(range(len(rev_m)))
ax3.set_xticklabels([m[:3] for m in MONTH_ORDER[:len(rev_m)]], fontsize=9)
for bar in bars3:
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{bar.get_height():.1f}M', ha='center', color=TEXT, fontsize=8)
style_ax(ax3, 'Total Revenue by Month (EUR Millions)', 'Month', 'Revenue (Millions)')

# ── [1,2] ADR vs Special Requests ────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
adr_sr = df.groupby('total_of_special_requests')['adr'].mean()
bars4  = ax4.bar(adr_sr.index, adr_sr.values,
                 color=PALETTE[:len(adr_sr)], edgecolor=DARK_BG, linewidth=0.3, width=0.6)
for bar in bars4:
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{bar.get_height():.0f}', ha='center', color=TEXT, fontsize=9)
style_ax(ax4, 'Avg ADR by No. Special Requests', 'Special Requests', 'ADR (EUR)')

plt.savefig('hotel_dashboard_p2.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Dashboard Page 2 saved.')

## 🌍 Section 7 — Dashboard Page 3: Market Segments & Cancellation Drivers

In [ ]:
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('🌍  Market Segments, Geography & Cancellation Drivers',
             fontsize=18, fontweight='bold', color=TEXT, y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.93, bottom=0.05, left=0.06, right=0.97)

# ── [0,0] Market Segment Bookings ────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ms  = df['market_segment'].value_counts()
ax1.barh(ms.index[::-1], ms.values[::-1], color=PALETTE[:len(ms)][::-1],
         edgecolor=DARK_BG, linewidth=0.3, height=0.65)
for i, (idx, val) in enumerate(zip(ms.index[::-1], ms.values[::-1])):
    ax1.text(val+30, i, f'{val:,}', va='center', color=TEXT, fontsize=8.5)
ax1.set_xlim(0, ms.max()*1.18)
style_ax(ax1, 'Bookings by Market Segment', 'Count', '')

# ── [0,1] Cancellation by Market Segment ─────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ms_c = df.groupby('market_segment')['is_canceled'].mean().sort_values()*100
colors2 = [RED if v>50 else ORANGE if v>35 else TEAL for v in ms_c.values]
ax2.barh(ms_c.index, ms_c.values, color=colors2, edgecolor=DARK_BG, linewidth=0.3, height=0.6)
ax2.axvline(cancel_rate, color=YELLOW, linewidth=1.5, linestyle='--',
            label=f'Overall {cancel_rate:.1f}%')
ax2.legend()
for i, val in enumerate(ms_c.values):
    ax2.text(val+0.3, i, f'{val:.1f}%', va='center', color=TEXT, fontsize=8.5)
style_ax(ax2, 'Cancel Rate by Market Segment', 'Cancel Rate (%)', '')

# ── [0,2] Top 10 Countries ────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
top_c = df['country'].value_counts().head(10)
ax3.barh(top_c.index[::-1], top_c.values[::-1],
         color=PALETTE[:10][::-1], edgecolor=DARK_BG, linewidth=0.3, height=0.65)
for i, val in enumerate(top_c.values[::-1]):
    ax3.text(val+15, i, f'{val:,}', va='center', color=TEXT, fontsize=8.5)
ax3.set_xlim(0, top_c.max()*1.18)
style_ax(ax3, 'Top 10 Countries by Bookings', 'Count', 'Country')

# ── [1,0] Lead Time Bins vs Cancellation ─────────────────────
ax4 = fig.add_subplot(gs[1, 0])
lt_bins = pd.cut(df['lead_time'],
                 bins=[0,30,60,90,120,180,240,365,600],
                 labels=['0-30','30-60','60-90','90-120','120-180','180-240','240-365','365+'])
lt_c = df.groupby(lt_bins, observed=True)['is_canceled'].mean()*100
colors4 = [GREEN if v<30 else ORANGE if v<45 else RED for v in lt_c.values]
ax4.bar(range(len(lt_c)), lt_c.values, color=colors4,
        edgecolor=DARK_BG, linewidth=0.3)
ax4.set_xticks(range(len(lt_c)))
ax4.set_xticklabels(lt_c.index, fontsize=8, rotation=25)
for i, val in enumerate(lt_c.values):
    ax4.text(i, val+0.4, f'{val:.1f}%', ha='center', color=TEXT, fontsize=8)
style_ax(ax4, 'Cancellation Rate by Lead Time', 'Lead Time Bucket', 'Cancel Rate (%)')

# ── [1,1] Special Requests vs Cancellation ───────────────────
ax5 = fig.add_subplot(gs[1, 1])
sr_c = df.groupby('total_of_special_requests')['is_canceled'].mean()*100
ax5.fill_between(sr_c.index, sr_c.values, alpha=0.2, color=PURPLE)
ax5.plot(sr_c.index, sr_c.values, color=PURPLE, marker='o', markersize=8,
         linewidth=2.5, markerfacecolor=PURPLE, markeredgecolor=DARK_BG, markeredgewidth=1.5)
for x, y in zip(sr_c.index, sr_c.values):
    ax5.text(x, y+0.5, f'{y:.1f}%', ha='center', color=TEXT, fontsize=9)
style_ax(ax5, 'Cancel Rate vs Special Requests', 'No. of Special Requests', 'Cancel Rate (%)')

# ── [1,2] Stay Duration by Guest Type ────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
stay_gt = df.groupby('is_repeated_guest')['total_stays'].mean()
bars6   = ax6.bar(['New Guest','Repeated Guest'], stay_gt.values,
                  color=[TEAL, ORANGE], edgecolor=DARK_BG, linewidth=0.4, width=0.45)
for bar in bars6:
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
             f'{bar.get_height():.2f} nts',
             ha='center', color=TEXT, fontsize=11, fontweight='bold')
style_ax(ax6, 'Avg Stay Duration by Guest Type', 'Guest Type', 'Avg Nights')

plt.savefig('hotel_dashboard_p3.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Dashboard Page 3 saved.')

## 📊 Section 8 — Dashboard Page 4: Advanced Analytics

In [ ]:
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('📊  Advanced Analytics — Heatmaps, Patterns & Room Analysis',
             fontsize=18, fontweight='bold', color=TEXT, y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.93, bottom=0.05, left=0.06, right=0.97)

# ── [0,0:2] Lead Time Boxplot by Market Segment ───────────────
ax1 = fig.add_subplot(gs[0, 0:2])
seg_order = df['market_segment'].value_counts().index.tolist()
groups    = [df[df['market_segment']==s]['lead_time'].values for s in seg_order]
bp = ax1.boxplot(groups, labels=seg_order, patch_artist=True, notch=False,
                 boxprops=dict(facecolor=SURFACE2, color=TEAL),
                 medianprops=dict(color=ORANGE, linewidth=2),
                 whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
                 flierprops=dict(marker='.', color=PINK, markersize=2, alpha=0.3))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.35)
ax1.tick_params(axis='x', rotation=20, labelsize=8.5)
style_ax(ax1, 'Lead Time Distribution by Market Segment', 'Market Segment', 'Lead Time (days)')

# ── [0,2] Room Type Distribution ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
rt   = df['reserved_room_type'].value_counts()
bars2 = ax2.bar(rt.index, rt.values, color=PALETTE[:len(rt)],
                edgecolor=DARK_BG, linewidth=0.3, width=0.65)
for bar in bars2:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
             str(int(bar.get_height())), ha='center', color=TEXT, fontsize=8.5)
style_ax(ax2, 'Bookings by Reserved Room Type', 'Room Type', 'Bookings')

# ── [1,0:2] Monthly Update Heatmap ───────────────────────────
ax3 = fig.add_subplot(gs[1, 0:2])
if 'arrival_date_year' in df.columns:
    hm = (df.groupby(['arrival_date_year','arrival_date_month'], observed=True)
            .size().unstack(fill_value=0))
    # keep only month columns that are valid month names
    valid_months = [m for m in MONTH_ORDER if m in hm.columns]
    hm = hm[valid_months]
    hm.columns = [m[:3] for m in hm.columns]
    sns.heatmap(hm, ax=ax3, cmap='Blues', annot=True, fmt='d',
                linewidths=0.5, linecolor=DARK_BG,
                annot_kws={'size':9, 'color':TEXT},
                cbar_kws={'shrink':0.8})
    ax3.set_facecolor(SURFACE)
    ax3.set_title('Bookings Heatmap: Year × Month', color=TEXT, fontsize=11, fontweight='bold', pad=8)
    ax3.tick_params(colors=MUTED, labelsize=9)
    ax3.set_xlabel('Month', color=MUTED, fontsize=9)
    ax3.set_ylabel('Year',  color=MUTED, fontsize=9)

# ── [1,2] Content Rating: Free vs Paid ───────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ct2 = pd.crosstab(df['customer_type'], df['is_canceled'], normalize='index')*100
bottom = np.zeros(len(ct2))
for col, color, label in zip(ct2.columns, [TEAL, RED], ['Not Canceled','Canceled']):
    ax4.bar(ct2.index, ct2[col], bottom=bottom, color=color,
            label=label, edgecolor=DARK_BG, linewidth=0.3)
    bottom += ct2[col].values
ax4.axhline(50, color=MUTED, linewidth=0.8, linestyle='--', alpha=0.5)
ax4.legend(); ax4.set_ylim(0, 110)
ax4.tick_params(axis='x', rotation=20, labelsize=8)
style_ax(ax4, 'Cancellation % by Customer Type', 'Customer Type', 'Percentage (%)')

plt.savefig('hotel_dashboard_p4.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Dashboard Page 4 saved.')

## 🔍 Section 9 — Detailed EDA Questions

In [ ]:
# Q1: Distribution by hotel, month & market segment
print('Q1 — Booking distribution (hotel × month × market segment):')
dist = df.groupby(['hotel','arrival_date_month','market_segment'],
                  observed=True).size().unstack(fill_value=0)
display(dist.head(10))

In [ ]:
# Q2: Cancellation rate vs lead time (scatter)
cancel_by_lt = df.groupby('lead_time')['is_canceled'].mean().reset_index()

plt.figure(figsize=(12, 5))
plt.scatter(cancel_by_lt['lead_time'], cancel_by_lt['is_canceled'],
            alpha=0.5, color=TEAL, s=18)
# Trend line
z = np.polyfit(cancel_by_lt['lead_time'], cancel_by_lt['is_canceled'], 1)
p = np.poly1d(z)
xs = np.linspace(0, cancel_by_lt['lead_time'].max(), 200)
plt.plot(xs, p(xs), color=RED, linewidth=2, linestyle='--', label='Trend')
plt.legend()
plt.title('Cancellation Rate vs Lead Time', color=TEXT, fontsize=12, fontweight='bold')
plt.xlabel('Lead Time (days)', color=MUTED)
plt.ylabel('Cancellation Rate', color=MUTED)
plt.tight_layout()
plt.show()

corr_val = np.corrcoef(cancel_by_lt['lead_time'], cancel_by_lt['is_canceled'])[0,1]
print(f'Pearson correlation (Lead Time vs Cancel Rate): {corr_val:.4f}')

In [ ]:
# Q3: ADR over time (continuous trend line)
if 'arrival_date_year' in df.columns:
    df['arrival_date_str'] = (df['arrival_date_year'].astype(str) + '-' +
                              df['arrival_date_month'].astype(str))
    adr_time = df.groupby(['arrival_date_year','arrival_date_month'],
                           observed=True)['adr'].mean().reset_index()
    adr_time = adr_time.sort_values(['arrival_date_year','arrival_date_month'])
    adr_time['idx'] = range(len(adr_time))

    plt.figure(figsize=(14, 5))
    plt.fill_between(adr_time['idx'], adr_time['adr'], alpha=0.2, color=BLUE)
    plt.plot(adr_time['idx'], adr_time['adr'], color=BLUE, linewidth=2)
    plt.title('Average Daily Rate (ADR) Over Time', color=TEXT, fontsize=12, fontweight='bold')
    plt.xlabel('Time Period', color=MUTED)
    plt.ylabel('ADR (EUR)', color=MUTED)
    plt.tight_layout()
    plt.show()

In [ ]:
# Q4: Cancellation by hotel type (detailed)
print('Q4 — Cancellation Rate by Hotel Type:')
print(df.groupby('hotel')['is_canceled'].agg(['mean','sum','count'])
        .rename(columns={'mean':'Cancel Rate','sum':'Canceled','count':'Total'})
        .assign(**{'Cancel Rate': lambda x: (x['Cancel Rate']*100).round(2).astype(str)+'%'}))

print(f'\nQ5 — Total canceled bookings: {total_canceled:,}')
print(f'Q6 — Most frequent arrival month: {top_month}')
print(f'Q7 — Average lead time: {avg_lead:.2f} days')
print(f'Q8 — Average ADR: €{avg_adr:.2f}')
print(f'Q9 — Average stay duration: {avg_stay:.2f} nights')
print(f'Q10 — Parking required: {free_pct:.2f}% of bookings')

In [ ]:
# Q11: Average stay duration by guest type
print('Q11 — Avg Stay Duration by Guest Type (0=New, 1=Repeat):')
print(df.groupby('is_repeated_guest')['total_stays'].mean().round(2))

# Q12: Top country
print(f'\nQ12 — Country with most bookings: {top_country}')
print(df['country'].value_counts().head(10))

# Q13: ADR by hotel type
print('\nQ13 — Average ADR by Hotel Type:')
print(df.groupby('hotel')['adr'].mean().round(2))

# Q14: Avg previous cancellations by hotel
if 'previous_cancellations' in df.columns:
    print('\nQ14 — Avg Previous Cancellations by Hotel:')
    print(df.groupby('hotel')['previous_cancellations'].mean().round(4))

In [ ]:
# Q15: Correlation — bookings changes vs special requests
corr_bc_sr = df['booking_changes'].corr(df['total_of_special_requests'])
print(f'Q15 — Correlation (Booking Changes vs Special Requests): {corr_bc_sr:.4f}')

avg_sr_by_chg = df.groupby('booking_changes')['total_of_special_requests'].mean()
print('\nAvg Special Requests by Booking Changes:')
print(avg_sr_by_chg.round(3))

# Q16: Highest revenue month
rev_m2 = df.groupby('arrival_date_month', observed=True)['revenue_per_booking'].sum()
best_month = rev_m2.idxmax()
print(f'\nQ16 — Highest Revenue Month: {best_month} (€{rev_m2.max():,.0f})')

## 💡 Section 10 — Key Insights & Business Recommendations

| # | Insight | Recommendation |
|---|---------|----------------|
| 1 | **City hotels cancel ~37% vs ~27% for resorts** | Apply stricter deposit policies for city hotel bookings; offer non-refundable discounts |
| 2 | **Lead time 365+ days → highest cancellation** | Require partial or full non-refundable deposits on bookings made 6+ months ahead |
| 3 | **Non-refundable deposits nearly eliminate cancellations** | Incentivise with 5–10% discount to shift guests towards non-refundable rates |
| 4 | **Online TAs drive most volume but elevated cancel rate** | Invest in direct booking perks (upgrades, free breakfast) to reduce OTA dependency |
| 5 | **3+ special requests → drastically lower cancel rate** | Proactively ask guests for special requests during booking to increase commitment |
| 6 | **July–August = peak ADR & Revenue** | Use dynamic pricing to maximise yield; plan staffing and maintenance in advance |
| 7 | **Repeat guests stay ~30% longer than new guests** | Invest in loyalty programme and post-stay emails with exclusive return offers |
| 8 | **Portugal (PRT) dominates bookings** | Localise marketing campaigns and offer Portuguese-language customer service channels |


In [ ]:
# ── Download all saved dashboard images ───────────────────────
try:
    from google.colab import files
    for fname in ['hotel_dashboard_p1.png','hotel_dashboard_p2.png',
                  'hotel_dashboard_p3.png','hotel_dashboard_p4.png']:
        files.download(fname)
    print('✅ All dashboard pages downloaded!')
except:
    print('Run in Google Colab to download. Files saved as hotel_dashboard_p1-4.png')